# Chapter 24 — Text-to-Speech and Speech Recognition Engines

Source: [Automate the Boring Stuff with Python, 3rd Ed. — Chapter 24](https://automatetheboringstuff.com/3e/chapter24.html)

**Setup note:** Needs `pip install pyttsx3 openai-whisper` (Whisper also needs ffmpeg installed on your system).

This notebook favors *reading* over *writing*: each section has a short explanation followed by real, runnable code. Read it, run it, tweak a value and re-run it — that's the fastest way to build reading fluency.

---


## 24.1 Basic text-to-speech with pyttsx3

`pyttsx3.init()` returns an Engine object wired to your OS's built-in speech engine. `.say(text)` queues speech; nothing is actually spoken until you call `.runAndWait()`, which blocks until speaking finishes.


In [1]:
# import pyttsx3
# engine = pyttsx3.init()
# engine.say('Hello. How are you doing?')
# engine.runAndWait()
print('say() queues text; runAndWait() actually speaks it and blocks until done.')


say() queues text; runAndWait() actually speaks it and blocks until done.


## 24.2 Adjusting voice, rate, and saving to a file

`.getProperty('rate' | 'volume' | 'voices')` reads current settings; `.setProperty(...)` changes them. `.save_to_file(text, path)` (followed by `runAndWait()`) renders speech to a `.wav` file instead of the speakers.


In [2]:
# import pyttsx3
# engine = pyttsx3.init()
# engine.setProperty('rate', 150)      # words per minute
# engine.setProperty('volume', 0.8)    # 0.0 to 1.0
# engine.save_to_file('Saved as audio.', 'demo.wav')
# engine.runAndWait()
print('save_to_file() + runAndWait() renders speech to disk instead of speakers.')


save_to_file() + runAndWait() renders speech to disk instead of speakers.


## 24.3 Speech-to-text with Whisper

OpenAI's Whisper model (used locally, no internet needed after the first download) transcribes audio files to text. `whisper.load_model('base')` loads a model of a given size/accuracy tradeoff; `.transcribe(path)` returns a dict whose `'text'` key holds the transcription.


In [3]:
# import whisper
# model = whisper.load_model('base')   # 'tiny' | 'base' | 'small' | 'medium' | 'large-v3'
# result = model.transcribe('hello.wav')
# print(result['text'])
print('Bigger models: slower but more accurate. base is a solid default for most uses.')


Bigger models: slower but more accurate. base is a solid default for most uses.


## 24.4 Downloading audio/video with yt-dlp, then transcribing with Whisper

The third-party `yt-dlp` module downloads video/audio from YouTube and hundreds of other sites — `yt_dlp.YoutubeDL(options).download([url])` fetches it, with options controlling format (e.g. audio-only) and output filename. The book pairs this directly with Chapter 24's Whisper speech-to-text: download a video's audio track with `yt-dlp`, then feed the resulting file straight into `whisper.transcribe()` to auto-generate a transcript/subtitles for it — a real end-to-end "any online video -> searchable text" pipeline.

**Environment note:** this sandbox's outbound network is restricted and can't reach YouTube (the request is blocked at the network boundary), so the download step can't run here. The code is real and correct — it works unmodified with a normal internet connection.

In [4]:
import yt_dlp

url = 'https://www.youtube.com/watch?v=jNQXAC9IVRw'   # "Me at the zoo", the first YouTube video ever
options = {
    'format': 'bestaudio/best',
    'outtmpl': 'downloaded_audio.%(ext)s',
    'quiet': True,
}

try:
    with yt_dlp.YoutubeDL(options) as ydl:
        info = ydl.extract_info(url, download=False)   # metadata only, to keep this demo light
        print('Title:', info.get('title'))
        print('Duration (s):', info.get('duration'))
    print()
    print('To actually download + transcribe: ydl.download([url]), then')
    print('  import whisper; model = whisper.load_model("base"); model.transcribe("downloaded_audio.m4a")')
except Exception as e:
    print(f'Could not reach YouTube from this sandbox ({type(e).__name__}) -- network restricted here.')
    print('This exact code works unmodified on a normal internet connection (e.g. in Cursor).')


Deprecated Feature: Support for Python version 3.10 has been deprecated. Please update to Python 3.11 or above


ERROR: [youtube] jNQXAC9IVRw: Unable to download API page: ('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')) (caused by ProxyError("('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden'))")); please report this issue on  https://github.com/yt-dlp/yt-dlp/issues?q= , filling out the appropriate issue template. Confirm you are on the latest version using  yt-dlp -U


Could not reach YouTube from this sandbox (DownloadError) -- network restricted here.
This exact code works unmodified on a normal internet connection (e.g. in Cursor).
